In [0]:
 # =============================================================================
# NOTEBOOK  : bronze_product_master
# PURPOSE   : Ingest product_master.parquet → bronze.product_master
# LAYER     : Bronze (raw ingestion — no transformation)
# FREQUENCY : Weekly + incremental (Autoloader availableNow)
# FORMAT    : Parquet
# NOTE      : expiry_date arrives as INT (days since epoch) — kept as-is
#             dimensions arrives as single STRING "LxWxH" — kept as-is
#             Both are parsed/split in silver, not here
# =============================================================================
 
# ── 0. IMPORTS & CONFIG ──────────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit
from utils.quality_checks import assert_not_empty
from utils.schema_registry import BRONZE_PRODUCT_MASTER_SCHEMA
 
from pyspark.sql.functions import current_timestamp, lit, col
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_PATH  = cfg["landing_paths"]["product_master"]
TARGET_TABLE = cfg["tables"]["bronze_product_master"]
CHECKPOINT   = cfg["checkpoint_paths"]["bronze_product_master"]
PIPELINE     = "bronze_product_master"

In [0]:
# ── 1. START AUDIT ────────────────────────────────────────────────────────────
run_id = start_audit(spark, PIPELINE, TARGET_TABLE)
 
try:
    # ── 2. READ (Autoloader — availableNow) ───────────────────────────────────
    # Parquet is typed — schema must match exactly what source sends
    # If source changes schema, Autoloader will detect and fail loudly
    # because mergeSchema is not set (we want loud failures, not silent evolution)
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
        .schema(BRONZE_PRODUCT_MASTER_SCHEMA)
        .load(SOURCE_PATH)
    )
 
    # ── 3. ADD AUDIT COLUMNS ──────────────────────────────────────────────────
    df = (
        df
        .withColumn("source_file",     col("_metadata.file_path"))
        .withColumn("ingested_at",     current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
    )
 
    # ── 4. WRITE ──────────────────────────────────────────────────────────────
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )
 
    query.awaitTermination()
 
    # ── 5. QUALITY CHECK ──────────────────────────────────────────────────────
    written_df   = spark.read.table(TARGET_TABLE) \
                       .where(f"pipeline_run_id = '{run_id}'")
    rows_written = written_df.count()
    # assert_not_empty(written_df, PIPELINE)
 
    print(f"[WRITE] {rows_written} rows written to {TARGET_TABLE}")
 
    # ── 6. END AUDIT (SUCCESS) ────────────────────────────────────────────────
    end_audit(spark, run_id, "SUCCESS", rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, run_id, "FAILED", error=str(e))
    raise
 